# 🎵 Bollywood Music Recommendation System — Model Training
A content-based recommender built with **TF-IDF** and **Cosine Similarity**.

This notebook trains the model and saves it to `../backend/artifacts/`, where the FastAPI backend loads it.

In [1]:
import os
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load the dataset
974 Bollywood songs collected from Spotify (2017–2023).

In [2]:
df = pd.read_csv('../data/Bollywood_Songs_With_Album_Genre.csv')
df.head()

,song_name,artist,release,lyrics,thumbnail,album,genre
0,Ek Haseena Thi Ek Deewana Tha,Yasser Desai,2017,Ek haseena thi ek deewana tha Kya kahun tumse ...,https://i.ytimg.com/vi/6obiArkHwAk/hqdefault.jpg,"Hue Bechain (From ""Ek Haseena Thi Ek Deewana T...","bollywood, hindi pop"
1,Hue Bechain,"Palak Muchhal, Yasser Desai",2017,Hue bechain pehli bar hmny raaz ye jana Mohabb...,https://i.ytimg.com/vi/6obiArkHwAk/hqdefault.jpg,"Hue Bechain (From ""Ek Haseena Thi Ek Deewana T...","bollywood, hindi pop"
2,Hanste Hanste,"Palak Muchhal, Yasser Desai",2017,Hanste hanste ro diye tum Kis mushkil mein kho...,https://i.ytimg.com/vi/rVKX19fNV0A/hqdefault.jpg,"Hanste Hanste (From ""Ek Haseena Thi Ek Deewana...","bollywood, hindi pop"
3,Nain,"Palak Muchhal, Yasser Desai",2017,Kuchh sawaal pyar ke lab pe hai ruke ruke Jaad...,https://i.ytimg.com/vi/ZHbbcy7u9Rk/hqdefault.jpg,"Nain (From ""Ek Haseena Thi Ek Deewana Tha"")","bollywood, hindi pop"
4,Aankhon Mein Aansoon,"Palak Muchhal, Yasser Desai",2017,Aankhon mein aansoo leke hoton se muskuraye Aa...,https://i.ytimg.com/vi/izP81UySf0Y/hqdefault.jpg,Unknown,Unknown


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   song_name  974 non-null    str  
 1   artist     974 non-null    str  
 2   release    974 non-null    int64
 3   lyrics     974 non-null    str  
 4   thumbnail  974 non-null    str  
 5   album      974 non-null    str  
 6   genre      974 non-null    str  
dtypes: int64(1), str(6)
memory usage: 1.2 MB


## 2. Check data quality
Look for null values and duplicate rows, then check the shape.

In [4]:
print('Null values per column:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('Shape:', df.shape)

Null values per column:
 song_name    0
artist       0
release      0
lyrics       0
thumbnail    0
album        0
genre        0
dtype: int64

Duplicate rows: 0
Shape: (974, 7)


## 3. Combine the useful text columns
Machine learning models work on numbers, so we first merge song name, artist, album and genre into one text column, then convert it to numbers.

In [5]:
df['combined'] = (
    df['song_name'].astype(str) + ' ' +
    df['artist'].astype(str) + ' ' +
    df['album'].astype(str) + ' ' +
    df['genre'].astype(str)
)
df['combined'].head()

0    Ek Haseena Thi Ek Deewana Tha Yasser Desai Hue...
1    Hue Bechain Palak Muchhal, Yasser Desai Hue Be...
2    Hanste Hanste Palak Muchhal, Yasser Desai Hans...
3    Nain Palak Muchhal, Yasser Desai Nain (From "E...
4    Aankhon Mein Aansoon Palak Muchhal, Yasser Des...
Name: combined, dtype: str

## 4. Convert text to vectors with TF-IDF
TF-IDF gives rare, meaningful words a higher weight and common words a lower weight. English stop words (is, the, are, …) are ignored.

In [6]:
tfidf = TfidfVectorizer(stop_words='english')
vectors = tfidf.fit_transform(df['combined'])
vectors.shape

(974, 2397)

## 5. Compute cosine similarity
Cosine similarity measures how close two song vectors are (1 = identical, 0 = unrelated).

In [7]:
similarity = cosine_similarity(vectors)
similarity.shape

(974, 974)

## 6. Recommendation function
Find the searched song's index, sort all songs by similarity to it, and return the top 5 (excluding the song itself).

In [8]:
def recommend(song, top_n=5):
    song = song.strip().lower()
    matches = df[df['song_name'].str.strip().str.lower() == song].index
    if len(matches) == 0:
        return []
    index = matches[0]
    scores = sorted(enumerate(similarity[index]), key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != index][:top_n]
    return [df.iloc[i]['song_name'] for i, _ in scores]

recommend('Ek Haseena Thi Ek Deewana Tha')

['Hue Bechain', 'Nain', 'Hanste Hanste', 'Ek Onkaar', 'Aasmaa']

## 7. Save the model with pickle
The backend only needs the song metadata and the similarity matrix (the vectorizer is saved too, for reuse).

In [9]:
out_dir = '../backend/artifacts'
os.makedirs(out_dir, exist_ok=True)

songs = df[['song_name', 'artist', 'release', 'album', 'genre', 'thumbnail']]
pickle.dump(songs, open(f'{out_dir}/songs_df.pkl', 'wb'))
pickle.dump(similarity, open(f'{out_dir}/similarity.pkl', 'wb'))
pickle.dump(tfidf, open(f'{out_dir}/tfidf_vectorizer.pkl', 'wb'))
print('Saved to', out_dir, ':', os.listdir(out_dir))

Saved to ../backend/artifacts : ['songs_df.pkl', 'tfidf_vectorizer.pkl', 'similarity.pkl']
